# 03a — Preparación no supervisada (conjuntos A/B/C/D)

Ejecuta el §5 + primera comprobación §7A de planning/plan_unsupervised_implementation.md.
Nota de convención: las transformaciones de ≤5 líneas viven aquí por legibilidad; todo lo que puede romperse en silencio (listas, matemática CLC, guardianes, config) vive en src/wildfire/processing/unsupervised.py con tests en 	ests/test_unsupervised.py.
Entrada: data/processed/expanded/firms_spain_final_lags_index.csv (30.412×60). Nunca irms_spain_final_normalized.csv (scaler descriptivo, plan §4 regla 3).
Salida: data/processed/unsupervised/features_{a,b,c}.csv + perfilado.csv (índice común).

## 03a.0 — Imports (celda única del cuaderno)

Todos los imports viven aquí y solo aquí: el resto de celdas no importa nada (convención del repo). Están la ciencia de datos habitual (`numpy`, `pandas`, `sklearn`), la E/S anclada a la raíz (`load_csv`/`save_csv`) y el módulo del bloque (`processing/unsupervised.py`), de donde salen las listas, las transformaciones y el guardián.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import RobustScaler, StandardScaler
from wildfire.config import PROJECT_ROOT
from wildfire.data.io import load_csv, save_csv
from wildfire.processing.pca_weather import WEATHER_PCA_BASE
from wildfire.processing.unsupervised import (
    ENTRADA_A, ENTRADA_B, ENTRADA_C_FIXED, ENTRADA_D, PERFILADO,
    add_wind_circular, apply_log_transforms, assert_feature_sets_clean,
    build_clc_environment, clc_environment_columns, resolve_set,
)

## 03a.1 — Cargar base y derivar viento circular

Primero se carga `firms_spain_final_lags_index.csv` (30.412 detecciones × 60 columnas), la base de todo el bloque 03. Después se deriva el viento circular (`wd_sin`/`wd_cos` con `AddWindCircular`, igual que en el PCA) y se comprueba que el par existe. La dirección cruda no entra en ningún conjunto: 359° y 1° son casi lo mismo y un escalado ingenuo rompería esa cercanía.

In [ ]:
df = load_csv("data/processed/expanded/firms_spain_final_lags_index.csv")
print(f"{len(df):,} rows × {df.shape[1]} columns")
df = add_wind_circular(df)
assert {"wd_sin", "wd_cos"} <= set(df.columns)

## 03a.2 — Mostrar los conjuntos cerrados y vigilarlos

Las listas A/B/C/D y PERFILADO se importan del módulo —fuente única de verdad— y se imprimen aquí para leer el pipeline entero sin abrir `src/`. Después pasa el guardián: si alguna columna de satélite (`frp`, `brightness`…) se hubiera colado en las entradas, la celda falla en cerrado en vez de contaminar el clustering. El contrato queda así: las entradas son causas cotidianas; el satélite y los presupuestos solo sirven para perfilar.

In [ ]:
for _name, _cols in [("A", ENTRADA_A), ("B", ENTRADA_B),
                     ("C_fixed", ENTRADA_C_FIXED), ("D", ENTRADA_D)]:
    print(f"{_name} ({len(_cols)}): {_cols}")
print(f"PERFILADO ({len(PERFILADO)})")
assert_feature_sets_clean(ENTRADA_A, ENTRADA_B, ENTRADA_C_FIXED, ENTRADA_D)
print("Guardián OK: sin columnas de satélite en las entradas.")

## 03a.3 — Transformaciones (≤5 líneas cada una, aquí por la nota de convención)

Tres gestos pequeños: `log1p` en precipitación, ráfagas y VPD, cuyas colas largas ya se midieron en 01b; la BLH se queda con su mediana global por decisión del plan; y las ocho direcciones CLC se colapsan en un entorno interpretable (% de vecinos iguales, nº de clases distintas, frecuencias) con el constructor que cubren los tests. Al final se enseña un resumen del entorno para inspeccionarlo a ojo.

In [ ]:
df = apply_log_transforms(df)
env = build_clc_environment(df)
df = pd.concat([df, env], axis=1)
env_cols = clc_environment_columns(env)
print(f"CLC environment: {len(env_cols)} columns ({len([c for c in env_cols if c.startswith('freq_vecina_')])} frequencies)")
display(env.describe().T[["mean", "std", "min", "max"]])

## 03a.4 — Resolver conjuntos y guardar (índice común)

Aquí se recorta el dataframe a cada lista cerrada (`resolve_set`: lo que falte avisa y se descarta, sin romper nada) y se guardan los cuatro ficheros en `data/processed/unsupervised/`: `features_a` (14, solo clima), `features_b` (19, + estacionalidad y hora), `features_c` (32, + suelo) y `perfilado` (31: satélite, presupuestos e índices). Los cuatro comparten índice para poder re-unirlos en 03b–03e sin ambigüedad.

In [ ]:
sets = {
    "features_a": resolve_set(df, ENTRADA_A, "A"),
    "features_b": resolve_set(df, ENTRADA_B, "B"),
    "features_c": resolve_set(df, [*ENTRADA_C_FIXED, *env_cols], "C"),
    "perfilado": resolve_set(df, PERFILADO, "PERFILADO"),
}
for _name, _frame in sets.items():
    save_csv(_frame, f"data/processed/unsupervised/{_name}.csv")
    print(f"{_name}: {_frame.shape}")

## 03a.5 — Primer número de robustez: ARI Standard vs Robust (plan §7A)

El primer número de robustez del bloque: agrupar lo mismo dos veces con KMeans (k=5, semilla 42), una con `StandardScaler` y otra con `RobustScaler`, sobre las 12 continuas meteorológicas, y medir el acuerdo con ARI (1 = idénticas, 0 = nada que ver). El plan exige más de 0,6 entre variantes. No es una elección de k —eso llega en 03b/03e—, sino una sonda: si saliera bajo, los grupos dependerían del scaler y habría que declararlo.

In [ ]:
base12 = resolve_set(df, WEATHER_PCA_BASE, "BASE12")
std_labels = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(StandardScaler().fit_transform(base12))
rob_labels = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(RobustScaler().fit_transform(base12))
ari = adjusted_rand_score(std_labels, rob_labels)
print(f"ARI(Standard, Robust) @ k=5: {ari:.3f} (listón plan §7A: > 0,6 entre variantes)")